# Aula 24: Árvore de Atingibilidade e Cobertura em Redes de Petri — Controle de Concorrência do Visão-AGV

## 1. Visão Geral da Aula

Nesta aula, aplicaremos **Redes de Petri (Petri Nets)** para modelar sistemas de eventos discretos e analisar estados de concorrência. Utilizaremos a **Árvore de Atingibilidade** para prever cenários operacionais e evitar *deadlocks* (impasses) na malha de roteamento do AGV.

### Contexto no Projeto Visão-AGV
Na intralogística, quando múltiplos AGVs operam simultaneamente ou quando um AGV interage com estações automatizadas (ex: braços robóticos, esteiras), ocorrem eventos paralelos e compartilhamento de recursos críticos.

Vamos modelar um cenário clássico de **Cruzamento Logístico (Zona de Exclusão Mútua)**:
- **Lugares (Places):** Representam os estados (ex: AGV_Aguardando, Zona_Livre).
- **Transições (Transitions):** Representam os eventos (ex: AGV_Entra_Zona, AGV_Sair_Zona).
- **Marcação (Marking):** O número de *tokens* em cada lugar, indicando o estado atual do sistema.


In [ ]:
# Implementação de uma Rede de Petri e Algoritmo de Árvore de Atingibilidade

class RedePetri:
    def __init__(self, lugares, transicoes):
        """
        lugares: lista de nomes dos lugares
        transicoes: dict {nome_transicao: {'in': {lugar: peso}, 'out': {lugar: peso}}}
        """
        self.lugares = lugares
        self.transicoes = transicoes

    def transicoes_habilitadas(self, marcacao):
        """Retorna as transições que podem ser disparadas na marcação atual."""
        habilitadas = []
        for nome_t, regras in self.transicoes.items():
            pode_disparar = True
            for lugar, peso in regras['in'].items():
                idx_lugar = self.lugares.index(lugar)
                if marcacao[idx_lugar] < peso:
                    pode_disparar = False
                    break
            if pode_disparar:
                habilitadas.append(nome_t)
        return habilitadas

    def disparar(self, marcacao, nome_t):
        """Gera uma nova marcação (estado) após disparar a transição."""
        nova_marcacao = list(marcacao)
        regras = self.transicoes[nome_t]
        
        # Consome tokens das entradas
        for lugar, peso in regras['in'].items():
            nova_marcacao[self.lugares.index(lugar)] -= peso
            
        # Produz tokens nas saídas
        for lugar, peso in regras['out'].items():
            nova_marcacao[self.lugares.index(lugar)] += peso
            
        return tuple(nova_marcacao)

print("Classe base da Rede de Petri criada com sucesso!")

## 2. Geração da Árvore de Atingibilidade
A árvore explora todos os estados (marcações) possíveis a partir do estado inicial. O algoritmo identifica estados repetidos para evitar loops infinitos.

In [ ]:
class NoAtingibilidade:
    def __init__(self, marcacao, pai=None, transicao_geradora=None):
        self.marcacao = marcacao
        self.pai = pai
        self.transicao_geradora = transicao_geradora
        self.filhos = []
        self.eh_duplicata = False
        self.eh_deadlock = False

def gerar_arvore_atingibilidade(rede, marcacao_inicial):
    raiz = NoAtingibilidade(marcacao_inicial)
    fronteira = [raiz]
    historico_marcacoes = {marcacao_inicial}
    todos_nos = [raiz]
    
    while fronteira:
        no_atual = fronteira.pop(0)
        transicoes_hab = rede.transicoes_habilitadas(no_atual.marcacao)
        
        if not transicoes_hab:
            no_atual.eh_deadlock = True
            continue
            
        for t in transicoes_hab:
            nova_marcacao = rede.disparar(no_atual.marcacao, t)
            novo_no = NoAtingibilidade(nova_marcacao, no_atual, t)
            no_atual.filhos.append(novo_no)
            todos_nos.append(novo_no)
            
            if nova_marcacao in historico_marcacoes:
                novo_no.eh_duplicata = True
            else:
                historico_marcacoes.add(nova_marcacao)
                fronteira.append(novo_no)
                
    return raiz, todos_nos

def imprimir_arvore(no, nivel=0):
    status = ""
    if no.eh_duplicata:
        status = " [ESTADO REPETIDO - FIM DE RAMO]"
    elif no.eh_deadlock:
        status = " [🚨 DEADLOCK DETECTADO]"
        
    caminho = f" --({no.transicao_geradora})--> " if no.transicao_geradora else ""
    print("  " * nivel + caminho + str(no.marcacao) + status)
    for filho in no.filhos:
        imprimir_arvore(filho, nivel + 1)

print("Algoritmo de expansão de estados inicializado.")

## 3. Simulação: Controle de Interseção Crítica
O Visão-AGV precisa atravessar um nó de trânsito (Cruzamento Logístico). Há um semáforo virtual (token de liberação) no SCADA que garante que apenas um AGV ocupe a zona simultaneamente.

In [ ]:
# Modelagem do Sistema Logístico
lugares = [
    'AGV_Aguardando',       # AGV pronto para entrar na zona crítica
    'Zona_Livre',           # Recurso de controle (Semáforo do SCADA)
    'AGV_Na_Zona',          # AGV operando/transitanto pela zona de interseção
    'Tarefa_Concluida'      # AGV liberou a zona e seguiu viagem
]

transicoes = {
    'T1_Entrar_Zona': {
        'in': {'AGV_Aguardando': 1, 'Zona_Livre': 1},
        'out': {'AGV_Na_Zona': 1}
    },
    'T2_Sair_Zona': {
        'in': {'AGV_Na_Zona': 1},
        'out': {'Zona_Livre': 1, 'Tarefa_Concluida': 1}
    }
}

rede_agv = RedePetri(lugares, transicoes)

# Cenário de Teste: 2 AGVs chegam simultaneamente no cruzamento
# Apenas 1 token na 'Zona_Livre' (recurso compartilhado)
marcacao_inicial = (2, 1, 0, 0) 

print(f"Ordem dos Lugares: {lugares}")
print(f"Estado Inicial: {marcacao_inicial} -> 2 AGVs aguardando, Zona Livre, 0 concluídos\n")

raiz_arvore, lista_estados = gerar_arvore_atingibilidade(rede_agv, marcacao_inicial)

print("=== ÁRVORE DE ATINGIBILIDADE ===")
imprimir_arvore(raiz_arvore)

## 4. Análise de Confiabilidade do Sistema

1. **Exclusão Mútua Garantida:** Se observarmos a árvore de estados, a posição de índice 2 (`AGV_Na_Zona`) **nunca ultrapassa o valor 1**. Isso prova matematicamente que colisões físicas no cruzamento logístico estão impossibilitadas.
2. **Prevenção de Impasses (Liveness):** A árvore demonstra que o sistema sempre consegue progredir até o estado `(0, 1, 0, 2)`, ou seja, os 2 AGVs concluem a travessia sem travamentos irreversíveis (*deadlocks*).
3. **Próximos Passos:** Esse modelo matemático será embutido no **SCADA-Core** para processar e liberar rotas seguras (zonas de transferência) para a frota de AGVs.